# Post-processing avec OPTICS

Data utilisé : pre-processing effectué avec algorithme de Louise (CSV) + drift correction avec ImageJ (cross-correlation)

In [14]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# Imports 
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from matplotlib.colors import hsv_to_rgb
import tifffile as tiff
import math
from sklearn.cluster import OPTICS
import matplotlib.gridspec as gridspec
from tqdm import tqdm

In [4]:
# Recuperation of data
data = pd.read_csv('image_Pos0_driftcorrected.csv',sep=',')
print(data.columns)

mask = tiff.imread("Mask.tif")    
mask = np.array(mask.transpose()) #Fiji écrit en y,x

Index(['frame', 'x [nm]', 'y [nm]', 'z [nm]', 'intensity', 'bkd', 'resnorm',
       'sigmax [nm]', 'sigmay [nm]', 'sigmaz [nm]', 'delta', 'deltaz1',
       'deltaz3', 'rho', 'rhoz1', 'rhoz3', 'deltaz', 'rhoz'],
      dtype='str')


In [ ]:
#Import data
frame = data['frame'].values
X = data[['x [nm]', 'y [nm]', 'z [nm]']].values
rho = data['rho'].values
delta = data['delta'].values
N_photons = data['intensity'].values
sigma = data[['sigmax [nm]', 'sigmay [nm]', 'sigmaz [nm]']].values

#Apply mask
mask_vect = (mask[(X[:, 0] / (120/5)).astype(int),(X[:, 1] / (120/5)).astype(int)] > 0)&(sigma[:,0] <= 240) & (sigma[:,1] <= 240) & (sigma[:,2] <= 720)
X_masked = X[mask_vect]
sigma_masked   = sigma[mask_vect]
rho_masked     = rho[mask_vect]
delta_masked   = delta[mask_vect]
frame_masked   = frame[mask_vect]

print('after masking free-hand roi & noisy outliers data set is ', len(X_masked), ' instead of', len(X))

after masking  134557  when original data set was  514722


In [15]:
#Arbitrary threshhold for merging localizations in consecutive frames 

def recursive_call(not_counted,i,X, rho, delta, frame, previous_index,th_lat=50, th_axial=75):
    #i is index of the frame we are in 
    #Let's create a mask that takes into account only the next frame and that looks for points at <th_lat and <th_ax
    mask = (frame==frame[i]+1) & ((X[:,0]-X[i,0])**2+(X[:,1]-X[i,1])**2<th_lat**2) & ((X[:,2]-X[i,2])**2<th_axial**2)
    #Is there at least one True value in mask?
    if (mask==True).any():
        not_counted[i] = False
        return recursive_call(not_counted, np.where(mask)[0][0], X, rho, delta, frame, previous_index=np.concatenate((previous_index, np.where(mask)[0])),th_lat=50, th_axial=75)
    else:
        return previous_index.astype(int)

In [ ]:
#Commençons par obtenir les std obtenus avec les threshholds arbitraires voir si ça tient la route
#Init
nb_id = len(X_masked[:,0])
not_counted = np.ones(nb_id, dtype=bool)
stdx = []
stdy = []
stdz = []

th_lat= 10
th_axial = 20 #Obtenus après calcul de std


for i in tqdm(range(nb_id), desc="Processing points"):
    if not_counted[i]:
        indices =  recursive_call(not_counted,i,X_masked, rho_masked, delta_masked, frame_masked, previous_index=np.array([i]),th_lat=th_lat, th_axial= th_axial)
        if len(indices)>1:
            #Pour le calcul de threshhold
            '''stdx.append(np.std([X_masked[j,0] for j in indices]))
            stdy.append(np.std([X_masked[j,1] for j in indices]))
            stdz.append(np.std([X_masked[j,2] for j in indices]))
            stdrho.append(np.std(rho_masked[indices]))
            stddelta.append(np.std(delta_masked[indices]))'''

            #Collapse les doublons into their average position, keep it at the last index, and erase the others by setting them to NaN.
            X_masked[indices[-1], 0] = np.mean(X_masked[indices, 0])
            X_masked[indices[:-1], 0] = np.nan
            X_masked[indices[-1], 1] = np.mean(X_masked[indices, 1])
            X_masked[indices[:-1], 1] = np.nan
            X_masked[indices[-1], 2] = np.mean(X_masked[indices, 2])
            X_masked[indices[:-1], 2] = np.nan

            sigma_masked[indices[-1], 0] = np.mean(sigma_masked[indices, 0])
            sigma_masked[indices[:-1], 0] = np.nan
            sigma_masked[indices[-1], 1] = np.mean(sigma_masked[indices, 1])
            sigma_masked[indices[:-1], 1] = np.nan
            sigma_masked[indices[-1], 2] = np.mean(sigma_masked[indices, 2])
            sigma_masked[indices[:-1], 2] = np.nan

            rho_masked[indices[-1]] = np.mean(rho_masked[indices])
            rho_masked[indices[:-1]] = np.nan

            delta_masked[indices[-1]] = np.mean(delta_masked[indices])
            delta_masked[indices[:-1]] = np.nan

            frame_masked[indices[-1]] = np.mean(frame_masked[indices])
            frame_masked[indices[:-1]] = np.nan

print('removed ', len(np.where(np.isnan(X_masked[:,0]))[0]), ' over ', len(X_masked[:,0]))

'''stdx = np.array(stdx)
stdy = np.array(stdy)
stdz = np.array(stdz)
stdrho = np.array(stdrho)
stddelta = np.array(stddelta)

print(np.mean(stdx))
print(np.mean(stdy))
print(np.mean(stdz))
print(np.mean(stdrho))
print(np.mean(stddelta))'''

X_treated = np.array([X_masked[~np.isnan(X_masked[:,0]), 0] , X_masked[~np.isnan(X_masked[:,1]), 1] ,X_masked[~np.isnan(X_masked[:,2]), 2]])
sigma_treated = np.array([sigma_masked[~np.isnan(sigma_masked[:,0]), 0] , sigma_masked[~np.isnan(sigma_masked[:,1]), 1] ,sigma_masked[~np.isnan(sigma_masked[:,2]), 2]])
rho_treated = rho_masked[~np.isnan(rho_masked)]
delta_treated = delta_masked[~np.isnan(delta_masked)]
frame_treated = frame_masked[~np.isnan(frame_masked)]



Processing points: 100%|██████████| 134557/134557 [13:03<00:00, 171.66it/s]

removed  1106  over  134557


In [21]:
#Petite visualisation 
plt.close('all')
plt.rcParams['figure.figsize'] = [12,12]
hues = rho_treated / 180.0
hsv_colors = np.stack((hues, np.ones_like(hues), np.ones_like(hues)), axis=1)
rgb_colors = hsv_to_rgb(hsv_colors)
plt.scatter(X_treated[0], X_treated[1], c=rgb_colors, s=0.01)
plt.axis('equal')

(np.float64(10581.711139901256),
 np.float64(16970.33135984956),
 np.float64(6421.15036260081),
 np.float64(14226.444490769556))